# PHIStruct: Improving phage-host interaction prediction at low sequence similarity settings using structure-aware protein embeddings

<b>Mark Edward M. Gonzales<sup>1, 2</sup>, Jennifer C. Ureta<sup>1, 2</sup> & Anish M.S. Shrestha<sup>1, 2</sup></b>

<sup>1</sup> Bioinformatics Lab, Advanced Research Institute for Informatics, Computing and Networking, De La Salle University, Manila 1004 <br>
<sup>2</sup> Department of Software Technology, College of Computer Studies, De La Salle University, Manila 1004

✉️ gonzales.markedward@gmail.com, jennifer.ureta@gmail.com, anish.shrestha@dlsu.edu.ph

<hr>

# 💡 Prerequisites

### Option 1: Download the prerequisite files
1. Download `consolidated.tar.gz` from this [link](https://drive.google.com/file/d/1yQSXwlb37dm2ZLXGJHdIM5vmrzwPAwvI/view?usp=sharing), and unzip it. This should result in a folder named `consolidated`. <br> Technically, this notebook only needs `consolidated/rbp_embeddings_saprot_struct_mask_relaxed_r3.csv` and `consolidated/rbp_embeddings_saprot_relaxed_r3.csv`.
1. Create a folder named `inphared` inside `data`, and save the extracted `consolidated` folder inside `data/inphared`. 
1. Download `fasta.tar.gz` from this [link](https://drive.google.com/file/d/1NMFR3JrrrCHLoCMQp2nia4dgtcXs5x05/view?usp=sharing), and unzip it. This should result in a folder named `fasta`. <br> Technically, this notebook only needs the `.clstr` files inside `fasta`.
1. Save the extracted `fasta` folder inside `data/inphared`.

### Option 2: Generate the prerequisite files yourself
1. If you have run `3.4. Data Consolidation (SaProt with Structure Masking).ipynb`, then `data/inphared/consolidated` should have already been populated with the prerequisite files.
1. Consolidate the sequences of the proteins with predicted structures into a single FASTA file. <br>
   For reproducibility, we provide our consolidated FASTA file [here](https://drive.google.com/file/d/1LTZte1f4lreQ5MXWeM-y2Mtp9z96pXS7/view?usp=sharing).
1. Generate the protein clusters by running CD-HIT on this FASTA file at a sequence similarity threshold of 100%, following the instructions [here](https://github.com/weizhongli/cdhit). 
1. Rename the resulting `.clstr` file to `complete-struct-100.fasta.clstr` and the resulting FASTA file (containing only the representative sequences) to `complete-struct-100.fasta`. 
1. Generate `complete-struct-80.fasta.clstr`, `complete-struct-60.fasta.clstr`, and `complete-struct-40.fasta.clstr` by running CD-HIT on `complete-struct-100.fasta` at sequence similarity thresholds of 80%, 60%, and 40%, respectively.
1. Create a folder named `fasta` inside `data/inphared`, and save the four `.clstr` files inside `data/inphared/fasta`.

### Resulting folder structure

`experiments` (parent folder of this notebook) <br> 
↳ `data` <br>
&nbsp; &nbsp;↳ `inphared` <br>
&nbsp; &nbsp;&nbsp; &nbsp; ↳ `consolidated` <br>
&nbsp; &nbsp;&nbsp; &nbsp;&nbsp; &nbsp; ↳ `rbp_embeddings_saprot_relaxed_r3.csv` <br>
&nbsp; &nbsp;&nbsp; &nbsp;&nbsp; &nbsp; ↳ `rbp_embeddings_saprot_struct_mask_relaxed_r3.csv` <br>
&nbsp; &nbsp;&nbsp; &nbsp; ↳ `fasta` <br>
&nbsp; &nbsp;&nbsp; &nbsp;&nbsp; &nbsp; ↳ `complete-struct-100.fasta.clstr` <br>
&nbsp; &nbsp;&nbsp; &nbsp;&nbsp; &nbsp; ↳ `complete-struct-80.fasta.clstr` <br>
&nbsp; &nbsp;&nbsp; &nbsp;&nbsp; &nbsp; ↳ `complete-struct-40.fasta.clstr` <br>
&nbsp; &nbsp;&nbsp; &nbsp;&nbsp; &nbsp; ↳ `complete-struct-60.fasta.clstr` <br>
↳ `5.7. Benchmarking - Classifier Building & Evaluation (SaProt with Structure Masking).ipynb` (this notebook) <br>

<hr>

# 📁 Output files

The output files (i.e., the results of evaluating the model's performance) &mdash; which are saved in `temp/results` &mdash; are already included when the repository was cloned. <br>

<hr>

# Part I: Preliminaries

Import the necessary libraries and modules.

In [1]:
import math
import pickle
import os
import warnings

import pandas as pd
import numpy as np
import sklearn

import ConstantsUtil
import ClassificationUtil

%load_ext autoreload
%autoreload 2

In [2]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", 50)

pd.options.mode.chained_assignment = None

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", category=sklearn.exceptions.UndefinedMetricWarning
    )

In [3]:
constants = ConstantsUtil.ConstantsUtil()
util = ClassificationUtil.ClassificationUtil()

<hr>

# Part II: Classifier Building and Evaluation

Train a multilayer perceptron, and evaluate its performance at different train-versus-test similarity and confidence thresholds.

In [4]:
models = list(constants.SAPROT_STRUCT_MASK_PLM.keys())

for similarity in range(100, 39, -20):
    for model in models:
        model = model.lower()
        df, df_all, protein_clusters = util.filter_proteins_based_on_struct_and_seq_sim(
            f"{constants.INPHARED}/{constants.CONSOLIDATED}/rbp_embeddings_{model}.csv",
            f"{constants.INPHARED}/{constants.CONSOLIDATED}/rbp_embeddings_saprot_relaxed_r3.csv",
            f"{constants.INPHARED}/{constants.FASTA}/complete-struct-{similarity}.fasta.clstr",
        )

        include_proteins_in_cluster = True
        if similarity == 100:
            include_proteins_in_cluster = False

        print(f"*** {model}, similarity = {similarity}% ***")
        util.classify(
            df,
            model + "-mlp-eskapee-smotetomek",
            similarity,
            genus=[
                "enterococcus",
                "staphylococcus",
                "klebsiella",
                "acinetobacter",
                "pseudomonas",
                "enterobacter",
                "escherichia",
            ],
            feature_columns=[f"s{i}" for i in range(1, 1281)],
            include_proteins_in_cluster=include_proteins_in_cluster,
            rbp_embeddings_all=df_all,
            protein_clusters=protein_clusters,
            undersample_others=True,
            oversample_technique="SMOTETomek",
            model="MLP",
        )

*** saprot_struct_mask_relaxed_r3, similarity = 100% ***
Constructing training and test sets...
Training set shape: (16944, 1280)
Test set shape: (2340, 1280)
Training the model...
Saving evaluation results...
Confidence threshold k: 0.0%
                precision    recall  f1-score   support

 acinetobacter     0.8602    0.7207    0.7843       111
  enterobacter     0.3234    0.4655    0.3816       116
  enterococcus     0.8679    0.9020    0.8846        51
   escherichia     0.8889    0.8000    0.8421      1040
    klebsiella     0.7678    0.8894    0.8241       461
        others     0.0000    0.0000    0.0000        51
   pseudomonas     0.8299    0.9237    0.8743       354
staphylococcus     0.9264    0.9679    0.9467       156

      accuracy                         0.8120      2340
     macro avg     0.6831    0.7087    0.6922      2340
  weighted avg     0.8094    0.8120    0.8074      2340

Confidence threshold k: 10.0%
                precision    recall  f1-score   support


Confidence threshold k: 10.0%
                precision    recall  f1-score   support

 acinetobacter     0.5250    0.5780    0.5502       109
  enterobacter     0.1848    0.2036    0.1937       167
  enterococcus     0.6304    0.4833    0.5472        60
   escherichia     0.7375    0.7106    0.7238      1344
    klebsiella     0.5917    0.5832    0.5874       559
        others     0.0106    0.0167    0.0130        60
   pseudomonas     0.6366    0.6239    0.6302       351
staphylococcus     0.8191    0.8953    0.8556       172

      accuracy                         0.6311      2822
     macro avg     0.5170    0.5118    0.5126      2822
  weighted avg     0.6424    0.6311    0.6362      2822

Confidence threshold k: 20.0%
                precision    recall  f1-score   support

 acinetobacter     0.5398    0.5596    0.5495       109
  enterobacter     0.1921    0.2036    0.1977       167
  enterococcus     0.6444    0.4833    0.5524        60
   escherichia     0.7449    0.7016    0

Confidence threshold k: 20.0%
                precision    recall  f1-score   support

 acinetobacter     0.3503    0.5124    0.4161       121
  enterobacter     0.1034    0.1684    0.1282       196
  enterococcus     0.5672    0.7755    0.6552        49
   escherichia     0.6816    0.5469    0.6068      1589
    klebsiella     0.5000    0.4405    0.4684       588
        others     0.0127    0.0612    0.0211        49
   pseudomonas     0.5813    0.6015    0.5912       404
staphylococcus     0.8531    0.7771    0.8133       157

      accuracy                         0.5167      3153
     macro avg     0.4562    0.4854    0.4625      3153
  weighted avg     0.5826    0.5167    0.5439      3153

Confidence threshold k: 30.0%
                precision    recall  f1-score   support

 acinetobacter     0.3554    0.4876    0.4111       121
  enterobacter     0.1077    0.1633    0.1298       196
  enterococcus     0.5672    0.7755    0.6552        49
   escherichia     0.6801    0.5205    0

Confidence threshold k: 30.0%
                precision    recall  f1-score   support

 acinetobacter     0.1871    0.5843    0.2834        89
  enterobacter     0.1589    0.1111    0.1308       216
  enterococcus     0.5455    0.7317    0.6250        41
   escherichia     0.7704    0.5306    0.6284      1796
    klebsiella     0.4561    0.3643    0.4051       527
        others     0.0113    0.1220    0.0207        41
   pseudomonas     0.4071    0.5916    0.4823       311
staphylococcus     0.8193    0.7473    0.7816       182

      accuracy                         0.4920      3203
     macro avg     0.4194    0.4729    0.4197      3203
  weighted avg     0.6161    0.4920    0.5352      3203

Confidence threshold k: 40.0%
                precision    recall  f1-score   support

 acinetobacter     0.1955    0.5843    0.2930        89
  enterobacter     0.1679    0.1065    0.1303       216
  enterococcus     0.5686    0.7073    0.6304        41
   escherichia     0.7763    0.5100    0